# Bike Sharing (hourly) — Preprocessing for Model Training

Steps:
1. Load data
2. Feature: `temp_avg` = average of `temp` and `atemp`; drop `temp`, `atemp`
3. Drop `weekday`, `season`, `dteday`, `instant`, `casual`, `registered`
4. Replace `hum == 0` with the mean humidity (computed on the training set only)
5. Feature: `low_demand_hour` — 1 for hours 23, 0, 1, 2, 3, 4, 5 (11 pm to 6 am, avg cnt < 100), else 0
6. Chronological train/test split (time-series data, no shuffling) + `TimeSeriesSplit` for cross-validation
7. Save model-ready files

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

df = pd.read_csv('hour.csv', parse_dates=['dteday'])
print(df.shape)
print(df.head().to_string())

(17379, 17)
   instant     dteday  season  yr  mnth  hr  holiday  weekday  workingday  weathersit  temp   atemp   hum  windspeed  casual  registered  cnt
0        1 2011-01-01       1   0     1   0        0        6           0           1  0.24  0.2879  0.81        0.0       3          13   16
1        2 2011-01-01       1   0     1   1        0        6           0           1  0.22  0.2727  0.80        0.0       8          32   40
2        3 2011-01-01       1   0     1   2        0        6           0           1  0.22  0.2727  0.80        0.0       5          27   32
3        4 2011-01-01       1   0     1   3        0        6           0           1  0.24  0.2879  0.75        0.0       3          10   13
4        5 2011-01-01       1   0     1   4        0        6           0           1  0.24  0.2879  0.75        0.0       0           1    1


## Check the insight: average `cnt` by hour

In [2]:
hourly = df.groupby('hr')['cnt'].mean().round(1)
print(hourly.to_string())
print('\nHours with avg cnt < 100:', list(hourly[hourly < 100].index))

hr
0      53.9
1      33.4
2      22.9
3      11.7
4       6.4
5      19.9
6      76.0
7     212.1
8     359.0
9     219.3
10    173.7
11    208.1
12    253.3
13    253.7
14    240.9
15    251.2
16    312.0
17    461.5
18    425.5
19    311.5
20    226.0
21    172.3
22    131.3
23     87.8

Hours with avg cnt < 100: [0, 1, 2, 3, 4, 5, 6, 23]


## Sort chronologically
The rows must be in time order before splitting, so that the test set is strictly *after* the training set.

In [3]:
df = df.sort_values(['dteday', 'hr']).reset_index(drop=True)
print('From', df['dteday'].min().date(), 'to', df['dteday'].max().date())

From 2011-01-01 to 2012-12-31


## Step 2 — `temp_avg` feature, drop `temp` and `atemp`

In [4]:
df['temp_avg'] = (df['temp'] + df['atemp']) / 2
df = df.drop(columns=['temp', 'atemp'])
print(df[['temp_avg']].describe().T)

            count      mean       std   min     25%     50%     75%   max
temp_avg  17379.0  0.486381  0.181643  0.01  0.3303  0.4924  0.6306  0.98


## Step 3 — Drop unneeded columns

In [5]:
df = df.drop(columns=['weekday', 'season', 'instant', 'casual', 'registered'])
# keep dteday for now only to report split dates; dropped right after splitting
print(df.columns.tolist())

['dteday', 'yr', 'mnth', 'hr', 'holiday', 'workingday', 'weathersit', 'hum', 'windspeed', 'cnt', 'temp_avg']


## Step 5 — `low_demand_hour` feature (night 11 pm to 6 am)

In [6]:
LOW_DEMAND_HOURS = [23, 0, 1, 2, 3, 4, 5]
df['low_demand_hour'] = df['hr'].isin(LOW_DEMAND_HOURS).astype(int)
print(df.groupby('low_demand_hour')['cnt'].agg(['mean', 'count']).round(1))

                  mean  count
low_demand_hour              
0                252.3  12375
1                 34.1   5004


## Step 6 — Chronological train/test split
Last 20% of the timeline is the test set. Shuffling (`train_test_split` default) would leak future information into training.

In [7]:
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx].copy()
test = df.iloc[split_idx:].copy()
print(f"Train: {train['dteday'].min().date()} to {train['dteday'].max().date()}  ({len(train)} rows)")
print(f"Test:  {test['dteday'].min().date()} to {test['dteday'].max().date()}  ({len(test)} rows)")

train = train.drop(columns=['dteday'])
test = test.drop(columns=['dteday'])

Train: 2011-01-01 to 2012-08-07  (13903 rows)
Test:  2012-08-07 to 2012-12-31  (3476 rows)


## Step 4 — Replace `hum == 0` with the mean
The mean is computed from the **training set only** (ignoring the zero readings) and the same value is applied to the test set and later at deployment, so no test information leaks into training.

In [8]:
print('hum == 0 rows -> train:', (train['hum'] == 0).sum(), ' test:', (test['hum'] == 0).sum())
HUM_MEAN = train.loc[train['hum'] > 0, 'hum'].mean()
train.loc[train['hum'] == 0, 'hum'] = HUM_MEAN
test.loc[test['hum'] == 0, 'hum'] = HUM_MEAN
print('Humidity mean used:', round(HUM_MEAN, 4))
print('Remaining zeros:', (train['hum'] == 0).sum() + (test['hum'] == 0).sum())

hum == 0 rows -> train: 22  test: 0
Humidity mean used: 0.624
Remaining zeros: 0


## Features and target

In [9]:
TARGET = 'cnt'
X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_test, y_test = test.drop(columns=[TARGET]), test[TARGET]
print('Features:', X_train.columns.tolist())
print('X_train', X_train.shape, ' X_test', X_test.shape)
print('Missing values:', X_train.isna().sum().sum() + X_test.isna().sum().sum())
print(X_train.head().to_string())

Features: ['yr', 'mnth', 'hr', 'holiday', 'workingday', 'weathersit', 'hum', 'windspeed', 'temp_avg', 'low_demand_hour']
X_train (13903, 10)  X_test (3476, 10)
Missing values: 0
   yr  mnth  hr  holiday  workingday  weathersit   hum  windspeed  temp_avg  low_demand_hour
0   0     1   0        0           0           1  0.81        0.0   0.26395                1
1   0     1   1        0           0           1  0.80        0.0   0.24635                1
2   0     1   2        0           0           1  0.80        0.0   0.24635                1
3   0     1   3        0           0           1  0.75        0.0   0.26395                1
4   0     1   4        0           0           1  0.75        0.0   0.26395                1


## Time-series cross-validation folds
Use `tscv` in `GridSearchCV` / `cross_val_score` when tuning models — each fold trains on the past and validates on the next block.

In [10]:
tscv = TimeSeriesSplit(n_splits=5)
for i, (tr, va) in enumerate(tscv.split(X_train), 1):
    print(f'Fold {i}: train rows 0-{tr[-1]}  ->  validate rows {va[0]}-{va[-1]}')

Fold 1: train rows 0-2317  ->  validate rows 2318-4634
Fold 2: train rows 0-4634  ->  validate rows 4635-6951
Fold 3: train rows 0-6951  ->  validate rows 6952-9268
Fold 4: train rows 0-9268  ->  validate rows 9269-11585
Fold 5: train rows 0-11585  ->  validate rows 11586-13902


## Save model-ready data

In [11]:
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
print('Saved X_train.csv, X_test.csv, y_train.csv, y_test.csv')

Saved X_train.csv, X_test.csv, y_train.csv, y_test.csv
